In [ ]:
!dd if=/dev/urandom of=sample_file.bin bs=128 count=1
!openssl sha256 sample_file.bin

In [ ]:
import mmap

import ctypes
import json
import hashlib
import numpy as np
import random
import time

with open('sample_file.bin', 'r+b') as f:
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_WRITE) as mm:
        # Memory mapped file mm behaves like a bytearray
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash before sort: {sha256_hash}')

        # Treat memory mapped file bytes as integers array in memory.
        # Sorting them.
        int_array = np.frombuffer(mm, dtype=np.uint32)
        print('Bytes as array of garbage 32 unsigned integers')
        print(int_array[:32]) # Unsorted
        int_array.sort()
        print('Bytes as array of sorted 32 unsigned integers')
        print(int_array[:32]) # Sorted
        mm.flush()
        
        # numpy array must be closed before memory mapped file.
        # BufferError: cannot close exported pointers exist
        del int_array

        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash after sort: {sha256_hash}')

In [ ]:
with open('sample_file.bin', 'rb') as f:
    file_content = f.read()
    sha256_hash = hashlib.sha256(file_content).hexdigest()
    print(f'Checking that file content changed indeed: {sha256_hash}')
    
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_READ) as mm:
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(f'Hash computed from memory mapped file: {sha256_hash}')

        try:
            mm[0] = b'x'
            assert('Cannot write to read-only mmap')
        except TypeError as e:
            #TypeError: mmap can't modify a readonly memory map.
            print('Exception as expected: ', e)

In [ ]:
with open('sample_file.bin', 'r+b') as f:
    with mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_WRITE) as mm:
        # Memory mapped file mm behaves like a bytearray
        sha256_hash = hashlib.sha256(mm).hexdigest()
        print(sha256_hash)

        # Treating buffer as signed integers (~ half of them will be negative)
        IntArrayType = ctypes.c_int * (len(mm) // ctypes.sizeof(ctypes.c_int))
        int_array = IntArrayType.from_buffer(mm)
        
        j = json.dumps({'numbers' : [n for n in int_array]}, indent=4).encode('utf-8')
        print('Previously sorted 32 bit unsigned integers treated as signed')
        print(j)

        # Cannot resize memory mapped file without closing integers arrays still holding a reference.
        # BufferError: cannot close exported pointers exist
        del int_array

        # Deceptively called function "truncate" works just fine to grow the file size forward. 
        f.truncate(len(j))

        
        mm.resize(len(j))
        mm[0:] = j
        mm.flush()

In [ ]:
with open('sample_file.bin', 'r') as f:
    j = json.load(f)
    print('Reading presumably json file back')
    print(j)

In [ ]:
import subprocess

# A tiny C library with one function. fflush so the C-level write to stdout reaches
# the notebook before Python's own buffered output.
c_source = r'''
//#include <stdio.h>

int hello_from_mmap(void) {
    //puts("Hello from mmap!");
    //fflush(stdout);
    return 42;
}
'''
with open('hello_mmap.c', 'w') as f:
    f.write(c_source)

# Compile a position independent shared library
subprocess.run(['cc', '-shared', '-fPIC', '-O2', '-o', 'libhello_mmap.so', 'hello_mmap.c'], check=True)

def get_function_offset(function_name):
    offset = subprocess.run(['bash', '-c', f'objdump -d ./libhello_mmap.so | grep "{function_name}" | cut -d" " -f1'],capture_output=True, check=True).stdout
    offset = offset.decode('utf-8').strip()
    print(f'{function_name} offset: ', offset)
    return int(f'0x{offset}', 16)

# Open the .so with mmap, the way the dynamic loader itself maps shared libraries. A plain
# file mapping is not relocated, so its bytes can be read (here the ELF magic).
# dlopen normally mmaps and relocates the library's segments
# but we recklessly call a function from unitialized libray any way.
with open('libhello_mmap.so', 'r+b') as f:
    with mmap.mmap(f.fileno(), length=0, prot=(mmap.PROT_READ|mmap.PROT_WRITE|mmap.PROT_EXEC)) as so:
        print(f'Mapped {len(so)} bytes of libhello_mmap.so, ELF magic {bytes(so[:4])}')

        # Memory protection is opened for read, write and execute
        # because ctypes does not like the idea of taking address of buffer that is not writable.
        # TypeError: underlying buffer is not writable
        mmap_address = ctypes.addressof(ctypes.c_char.from_buffer(so))
        print('mmap_address: ', mmap_address)

        hello_from_mmap_offset = get_function_offset('<hello_from_mmap>')
        address = mmap_address + hello_from_mmap_offset
        print(f'hello_from_mmap at 0x{address:x} directly from memory mapped file')

        # DANGER! Reckless call from unitialized library that would result in a crash
        # if that function needed anything from C runtime or accessed global variables.
        hello_from_mmap = ctypes.CFUNCTYPE(ctypes.c_int)(address)
        result = hello_from_mmap()
        print(f'hello_from_mmap() returned {result}')

lib = ctypes.CDLL('./libhello_mmap.so')

# The symbol's runtime address, as a plain integer
address = ctypes.cast(lib.hello_from_mmap, ctypes.c_void_p).value
print(f'hello_from_mmap at 0x{address:x} via ctypes.CDLL')

# Build a callable from that raw address and call it by pointer: void hello_from_mmap(void)
hello_from_mmap = ctypes.CFUNCTYPE(ctypes.c_int)(address)
result = hello_from_mmap()
print(f'hello_from_mmap() returned {result}')